<a href="https://colab.research.google.com/github/AbhiGen/kid_LLM/blob/feat%2Fkids-nutrition-llm/projectPhase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import json
from pathlib import Path

BASE_PATH = Path("/content/drive/MyDrive/nutrikid/datasets")

with open(BASE_PATH / "parent.json", "r") as f:
    parent_data = json.load(f)

print("Total parent examples:", len(parent_data))
print(parent_data[0])


Total parent examples: 545
{'question': 'What is a healthy breakfast for my 5-year-old?', 'answer': 'For a 5-year-old, offer a balanced breakfast such as ½ cup cooked whole-grain upma or poha, along with a small serving of curd and a fruit like banana or apple slices. Whole grains provide steady energy, and the curd adds protein. Include a fruit for fiber and hydration. Avoid packaged sugary cereals.'}


In [3]:
SYSTEM_PROMPT = (
    "You are NutriKid, a safe pediatric nutrition assistant. "
    "You never diagnose diseases or prescribe medicine. "
    "If a question involves illness, allergies, or serious symptoms, "
    "advise consulting a doctor."
)


In [4]:
def convert(data, role):
    converted = []
    for item in data:
        converted.append({
            "instruction": f"<ROLE={role}> {item['question']}",
            "output": item['answer'],
            "system": SYSTEM_PROMPT
        })
    return converted


In [5]:
all_data = []

roles = {
    "PARENT": "parent.json",
    "KID": "kid.json",
    "DOCTOR": "doctor.json",
    "SAFETY": "safety.json"
}

for role, filename in roles.items():
    with open(BASE_PATH / filename) as f:
        data = json.load(f)
        all_data.extend(convert(data, role))

print("Total examples:", len(all_data))


Total examples: 2972


In [6]:
import random

random.shuffle(all_data)

out_path = Path("/content/drive/MyDrive/nutrikid/train.jsonl")

with open(out_path, "w") as f:
    for row in all_data:
        f.write(json.dumps(row) + "\n")

print("Saved training file:", out_path)


Saved training file: /content/drive/MyDrive/nutrikid/train.jsonl


In [7]:
!pip install -q transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.4 MB/s eta 0:00:00


In [8]:
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True
)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

TypeError: Qwen2ForCausalLM.__init__() got an unexpected keyword argument 'load_in_4bit'

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
